# Task 02: Build a Combined Multi-tab App

Build a single Gradio `Blocks` app with two tabs:
- **Tab 1**: GLiNER NER (reuse `run_ner` from Task 01)
- **Tab 2**: GLiClass classification with `gr.Label` probabilities

## Parts
1. Implement `run_cls()` → `gr.Label` output
2. Build multi-tab `gr.Blocks` app combining both models
3. Add `gr.State` to remember custom labels between calls

In [ ]:
import json, os
import gradio as gr
from gliner import GLiNER
from gliclass import GLiClassModel, ZeroShotClassificationPipeline
from transformers import AutoTokenizer

fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))
with open(os.path.join(fixtures, "sample_texts.json")) as f:
    SAMPLE_TEXTS = json.load(f)

ENTITY_TYPES = ["malware", "vulnerability", "attack_technique", "affected_software", "threat_actor"]
ENTITY_COLORS = {
    "malware": "#ff6b6b",
    "vulnerability": "#ffa94d",
    "attack_technique": "#74c0fc",
    "affected_software": "#69db7c",
    "threat_actor": "#da77f2",
}
ATTACK_LABELS = ["ransomware", "phishing", "apt_attack", "ddos", "data_breach", "supply_chain_attack", "zero_day"]

print(f"✓ Loaded {len(SAMPLE_TEXTS)} sample texts")

## Setup: Load Both Models

In [ ]:
ner_model = GLiNER.from_pretrained("knowledgator/gliner-bi-edge-v2.0")
print("✓ GLiNER loaded")

cls_model = GLiClassModel.from_pretrained("knowledgator/gliclass-edge-v3.0")
cls_tokenizer = AutoTokenizer.from_pretrained("knowledgator/gliclass-edge-v3.0", add_prefix_space=True)
cls_pipeline = ZeroShotClassificationPipeline(
    cls_model, cls_tokenizer, classification_type='multi-label', device='cpu'
)
print("✓ GLiClass loaded")

In [ ]:
# run_ner from Task 01 — already implemented here
def run_ner(text: str, entity_types: list, threshold: float) -> list:
    if not text.strip() or not entity_types:
        return [(text, None)]
    entities = ner_model.predict_entities(text, entity_types, threshold=threshold)
    entities = sorted(entities, key=lambda e: e["start"])
    result = []
    cursor = 0
    for ent in entities:
        start, end, label = ent["start"], ent["end"], ent["label"]
        if start > cursor:
            result.append((text[cursor:start], None))
        result.append((text[start:end], label))
        cursor = end
    if cursor < len(text):
        result.append((text[cursor:], None))
    return result if result else [(text, None)]

## Part 1: Implement `run_cls`

Run the GLiClass pipeline and return a `{label: score}` dict compatible with `gr.Label`.

- `labels_str`: comma-separated string of labels (e.g. `"ransomware, phishing, ddos"`)
- Use `cls_pipeline(text, labels, threshold=0.0)[0]` to get results
- Return `{result['label']: float(result['score']) for result in results}`

In [ ]:
def run_cls(text: str, labels_str: str) -> dict:
    """
    Run GLiClass classification.

    Returns:
        dict mapping label → score (for gr.Label output)
    """
    # YOUR CODE HERE
    pass


# TEST - Do not modify
scores = run_cls(SAMPLE_TEXTS[0]['text'], ', '.join(ATTACK_LABELS))
assert isinstance(scores, dict)
assert len(scores) > 0
assert all(isinstance(v, float) for v in scores.values())
assert all(k in ATTACK_LABELS for k in scores.keys())
top_label = max(scores, key=scores.get)
print(f"✓ run_cls works: top label = '{top_label}' ({scores[top_label]:.3f})")
# Sample 0 is ransomware — expect ransomware or apt_attack to be top
assert top_label in {"ransomware", "apt_attack", "zero_day"}, f"Unexpected top label: {top_label}"

## Part 2: Build Multi-tab `gr.Blocks` App

Combine both models in a single `gr.Blocks` with two `gr.Tab`s:

**Tab 1 — NER**:
- `gr.Textbox` input
- `gr.CheckboxGroup` for entity types
- `gr.Slider` for threshold  
- `gr.HighlightedText` output with `color_map` and `show_legend=True`

**Tab 2 — Classification**:
- `gr.Textbox` input
- `gr.Textbox` for comma-separated labels (default: ATTACK_LABELS joined)
- `gr.Label` output with `num_top_classes=7`

Both tabs should have `gr.Examples` with at least 3 sample texts.

In [ ]:
# YOUR CODE HERE — build combined_app with gr.Blocks + gr.Tabs
# combined_app = ...
# combined_app.launch()

# TEST - Do not modify
assert 'combined_app' in dir(), "Variable 'combined_app' must be defined"
assert hasattr(combined_app, 'launch')
print("✓ combined_app defined")

## Part 3: Add `gr.State` for Custom Label Memory

In the Classification tab, add a `gr.State` that remembers the **last used labels** between calls.
If the user clears the labels textbox and clicks classify, restore the last non-empty value from state.

Logic:
```python
def run_cls_stateful(text, labels_str, saved_labels):
    if not labels_str.strip():
        labels_str = saved_labels  # restore from state
    else:
        saved_labels = labels_str  # update state
    scores = run_cls(text, labels_str)
    return scores, saved_labels  # return (output, new_state)
```

In [ ]:
# YOUR CODE HERE — rebuild combined_app_stateful with gr.State
# combined_app_stateful = ...
# combined_app_stateful.launch()

# TEST - Do not modify
assert 'combined_app_stateful' in dir(), "Variable 'combined_app_stateful' must be defined"

# Test stateful logic directly
def run_cls_stateful(text, labels_str, saved_labels):
    if not labels_str.strip():
        labels_str = saved_labels
    else:
        saved_labels = labels_str
    scores = run_cls(text, labels_str)
    return scores, saved_labels

default_labels = ', '.join(ATTACK_LABELS)
scores1, state1 = run_cls_stateful(SAMPLE_TEXTS[0]['text'], default_labels, "")
assert state1 == default_labels
scores2, state2 = run_cls_stateful(SAMPLE_TEXTS[1]['text'], "", state1)  # empty → use saved
assert state2 == default_labels
assert len(scores2) > 0
print("✓ Stateful logic works")